<div align="center">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajeevraibhatia/agent-harness-evals/blob/main/notebooks/m4_multi_agent.ipynb)
[![Course](https://img.shields.io/badge/Full%20Course-rajeevraibhatia.com-7c3aed)](https://rajeevraibhatia.com/curriculum/agent-harness-evals#module-4)

</div>

# Module 4 — Multi-Agent Orchestration Patterns

**Level:** Advanced | **Time:** ~1.5h | [Full reading →](https://rajeevraibhatia.com/curriculum/agent-harness-evals#module-4)

### What you'll build
A supervisor + 2 specialist agents with a producer-critic review loop, using raw OpenAI SDK.

### Key concepts
- **Supervisor/router**: stateless specialists, supervisor does routing + handoff
- **Producer-critic loop**: writer produces, critic reviews, writer revises — stopping criteria matter
- **Debate / multi-agent reasoning** (Du 2023): +10% on ARC-Challenge but N× cost vs reflection + tools
- **Deadlock patterns**: mutual wait, circular handoff — detect with same-specialist-N-times guard
- **Decision framework**: add a 2nd agent only when tool cardinality, system prompt, or eval cycle independence demands it

### Research refs
- Multi-Agent Debate — Du et al. (2023) https://arxiv.org/abs/2305.14325
- AutoGen — Wu et al. (2023) https://arxiv.org/abs/2308.08155

---

In [ ]:
!pip install openai --quiet

In [ ]:
import os, json
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ── Specialist agents ─────────────────────────────────────────────────────────

def run_specialist(system_prompt: str, task: str, model: str = "gpt-4o") -> str:
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": task}
        ]
    )
    return response.choices[0].message.content

SPECIALISTS = {
    "researcher": "You are a research specialist. Find and synthesize factual information. Be concise and cite sources when possible.",
    "writer": "You are a writing specialist. Produce clear, well-structured prose. Adapt tone to the audience.",
    "critic": "You are a quality critic. Review content for accuracy, clarity, and completeness. Return structured feedback: {strengths: [], weaknesses: [], verdict: pass|revise}."
}

# ── Supervisor / Router ───────────────────────────────────────────────────────

ROUTING_SCHEMA = {
    "type": "function",
    "function": {
        "name": "route_task",
        "description": "Route a task to the appropriate specialist.",
        "parameters": {
            "type": "object",
            "properties": {
                "specialist": {
                    "type": "string",
                    "enum": ["researcher", "writer", "critic"],
                    "description": "Which specialist to invoke."
                },
                "task": {
                    "type": "string",
                    "description": "The specific task for the specialist."
                },
                "reason": {
                    "type": "string",
                    "description": "Why this specialist is the right choice."
                }
            },
            "required": ["specialist", "task", "reason"]
        }
    }
}

def supervisor(user_request: str, context: str = "") -> dict:
    """Route a user request to the right specialist."""
    msg = f"User request: {user_request}"
    if context:
        msg += f"\n\nContext from previous steps:\n{context}"

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a supervisor that routes tasks to specialist agents. Always use the route_task tool."},
            {"role": "user", "content": msg}
        ],
        tools=[ROUTING_SCHEMA],
        tool_choice={"type": "function", "function": {"name": "route_task"}}
    )
    args = json.loads(response.choices[0].message.tool_calls[0].function.arguments)
    return args

# ── Producer-Critic Loop ──────────────────────────────────────────────────────

def producer_critic_loop(task: str, max_revisions: int = 2) -> str:
    """Writer produces, critic reviews, writer revises if needed."""
    draft = run_specialist(SPECIALISTS["writer"], task)
    print(f"  [draft] {draft[:100]}...")

    for revision in range(max_revisions):
        feedback_raw = run_specialist(SPECIALISTS["critic"], f"Review this content:\n\n{draft}")
        print(f"  [critique round {revision+1}] {feedback_raw[:100]}...")

        try:
            feedback = json.loads(feedback_raw)
            verdict = feedback.get("verdict", "pass")
        except json.JSONDecodeError:
            verdict = "pass" if "pass" in feedback_raw.lower() else "revise"

        if verdict == "pass":
            print(f"  [verdict] PASS after {revision+1} critique(s)")
            return draft

        # Revise
        draft = run_specialist(SPECIALISTS["writer"],
            f"Revise this draft based on feedback.\n\nOriginal task: {task}\n\nDraft:\n{draft}\n\nFeedback:\n{feedback_raw}")
        print(f"  [revised] {draft[:100]}...")

    return draft

# ── Orchestrated pipeline ─────────────────────────────────────────────────────

def multi_agent_pipeline(user_request: str) -> str:
    print(f"Request: {user_request}\n")
    context = ""
    results = []

    # Step 1: Research
    route = supervisor(user_request, context)
    print(f"[supervisor] → {route['specialist']}: {route['reason']}")
    research = run_specialist(SPECIALISTS["researcher"], route["task"])
    results.append(f"Research:\n{research}")
    context = "\n\n".join(results)

    # Step 2: Write with producer-critic
    print("\n[producer-critic loop]")
    write_task = f"Write a clear summary of these research findings for a technical audience:\n\n{research}"
    final = producer_critic_loop(write_task)
    return final

result = multi_agent_pipeline("Explain the key differences between RAG and fine-tuning for LLM applications.")
print(f"\n=== Final Output ===\n{result}")

## Exercise

Design a multi-agent topology for a product requiring research, code generation, and content writing.

> **Interview question:** *"Design a multi-agent system for legal document review. What agents do you need? How do you handle conflicts? How do you eval it?"*

In [ ]:
# Exercise: Multi-agent topology for a product that needs:
# (a) web research, (b) code generation, (c) content writing

# 1. Design the supervisor routing logic as a Python dict or state machine
# 2. Identify 2 potential deadlock scenarios and their mitigations
# 3. When would you add a 4th agent vs keeping it to 3?

# Deadlock scenario example:
# Writer waits for Researcher; Researcher waits for Writer to define scope.
# Mitigation: supervisor always breaks ties with explicit task decomposition.

routing_logic = {
    "research_question": "researcher",
    "write_copy": "writer",
    "generate_code": "coder",  # needs implementing
    "review_output": "critic"
}

# TODO: implement the "coder" specialist
# TODO: add deadlock detection (detect if same specialist called 3x in a row)
# TODO: add a "none_of_the_above" → escalate_to_human path